# Hepatic lobule dFBA — run it in one notebook
Builds and runs the PhysiCell + dFBA lobule model (zonation, vascular transport, ATP calibration) and shows the figures.
Works on **Google Colab** (Ubuntu VM), on a local Jupyter under **Ubuntu / WSL2**, and inside the **Docker image** of this repository.

| Step | What it does | Time |
|---|---|---|
| 1 | locate (or `git clone --depth 1`) the project | ~1 min |
| 2 | install system packages + Python modules (needs root; Colab and Docker have it) | ~2 min |
| 3 | compile | 3–6 min |
| 4 | **quick test**: 3 short runs, must end with `OK` (bit-level check against our published runs) | ~5–10 min |
| 5 | *(optional)* one full arm, e.g. the ATP-calibrated lobule | ~25 min |
| 6 | analysis + figures (or look at `results/reference/` without running anything) | <1 min |

> Exploratory model: see `README.md` and `docs/` for limits. Nothing here is a validated result.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MatheusAmorim7/HepatoModel-dFBA.git"
BRANCH   = "lobule-dfba-reproducible"
PROJ_REL = "sample_projects_intracellular/fba/hepatic_lobule_dfba"

def sh(cmd, cwd=None):
    """Run a shell command and stream its output (so long builds show progress)."""
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, executable="/bin/bash", stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end="")
    p.wait()
    if p.returncode != 0: raise RuntimeError(f"command failed (rc={p.returncode}): {cmd}")

# 1) find the project: already inside it, or clone only the latest commit of the branch (~80 MB instead of the full ~1 GB history)
here = Path.cwd().resolve()
PROJ = next((d for d in [here, *here.parents] if (d / "scripts/make_configs.py").exists() and (d / "main.cpp").exists()), None)
if PROJ is None:
    clone = Path.home() / "hepatomodel-dfba"
    if not clone.exists(): sh(f"git clone --depth 1 --branch {BRANCH} {REPO_URL} {clone}")
    PROJ = clone / PROJ_REL
print("project:", PROJ)

In [ ]:
# 2) system packages (apt, only if something is missing and root/sudo is available) and Python modules for THIS kernel
sh(f"PY={sys.executable} ./scripts/setup_ubuntu.sh", cwd=PROJ)

In [ ]:
# 3) compile (first build compiles the PhysiCell core: a few minutes)
sh(f"make -j$(nproc) PY={sys.executable}", cwd=PROJ)

In [ ]:
# 4) quick test: three short runs compared with the published runs. Every line must say OK.
sh(f"make quick PY={sys.executable}", cwd=PROJ)

In [ ]:
# 5) OPTIONAL full arm (~25 min). Set RUN_FULL_ARM = True to run it. Arms: see config/arms.json
RUN_FULL_ARM = False
ARM = "A3_s3_pal449"        # ATP-calibrated lobule with prescribed zonation (6 simulated minutes)
if RUN_FULL_ARM:
    sh(f"make run ARM={ARM} PY={sys.executable}", cwd=PROJ)
    sh(f"{sys.executable} scripts/analyze.py {ARM} && {sys.executable} scripts/score.py && {sys.executable} scripts/make_figures.py", cwd=PROJ)

In [ ]:
# 6) figures: yours (results/figures) if you ran an arm, otherwise the reference figures from our original runs
from IPython.display import Image, display
for folder in (PROJ / "results/figures", PROJ / "results/reference/figures"):
    pngs = sorted(folder.glob("*.png"))
    if pngs:
        print("showing", folder.relative_to(PROJ))
        for f in pngs[:6]: print(f.name); display(Image(filename=str(f), width=900))
        break

## Next
* `make reproduce-zonation`, `make reproduce-emergent`, `make reproduce-atp` (see `README.md`) rerun a whole family of arms (hours).
* `make studio` converts your runs to the standard PhysiCell output so **PhysiCell Studio** can open them (see `docs/STUDIO_PT.md`).
* Compare your `results/SCORES.json` with `results/reference/`.